In [16]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [17]:
html = open(r"C:\Users\merfc\Downloads\veturilo5.html", encoding="utf-8").read()
html2 = open(r"C:\Users\merfc\Downloads\veturilo6.html", encoding="utf-8").read()

In [18]:
def reader(plik): 
    soup = BeautifulSoup(plik, "html.parser")
    
    rows = soup.find_all("div", role="row")
    trips = []
    for i in rows:
        start_place= i.find("div", {"data-field": "startPlace"})
        if not start_place: continue
        end_place = i.find("div", {"data-field": "endPlace"})
        start_time = i.find("div", {"data-field": "startTime"})
        end_time = i.find("div", {"data-field": "endTime"})
        duration = i.find("div", {"data-field": "duration"})
        #bike_type = i.find("div", {"data-field": "bikeType"})
    
        if start_place and end_place:
            trips.append({
                "start_place" : start_place.text.strip(),
                "end_place" : end_place.text.strip() if end_place else None,
                "duration": duration.text.strip() if duration else None,
                "start_time": start_time.text.strip() if start_time else None,
                "end_time": end_time.text.strip() if end_time else None
                #"bike_type": bike_type.text.strip() if bike_type else None,
            })
    return trips

In [19]:
def reader2(plik):
    soup = BeautifulSoup(plik,"html.parser")

    rows = soup.find_all("div",role="row")
    trips = []
    for i in rows:
        bike_type=i.find("div",{"data-field": "bikeType"})
        start_time = i.find("div", {"data-field": "startTime"})
        if bike_type:
            trips.append({
                "bike_type" : bike_type.text.strip(),
                "start_time": start_time.text.strip() if start_time else None
                
            })
    return trips
        

In [20]:
trips = reader(html)

In [21]:
df1 = pd.DataFrame(trips)

In [22]:
trips2 = reader2(html2)

In [23]:
df2 = pd.DataFrame(trips2)

In [24]:
df = pd.merge(df1, df2, on='start_time')

In [25]:
df = df.drop(0)

In [26]:
df.drop_duplicates(inplace=True)
df = df.reset_index(drop=True)


In [27]:
moment_startu = pd.to_datetime(df["start_time"])
moment_konca = pd.to_datetime(df["end_time"])
df["start_date"] = moment_startu.dt.date
df["start_time"] = moment_startu.dt.time
df["end_time"] = moment_konca.dt.time
df["end_date"] = moment_konca.dt.date

C:\Users\merfc\AppData\Local\Temp\ipykernel_16248\3048909370.py:1: UserWarning: Parsing dates in %d.%m.%Y, %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  moment_startu = pd.to_datetime(df["start_time"])
C:\Users\merfc\AppData\Local\Temp\ipykernel_16248\3048909370.py:2: UserWarning: Parsing dates in %d.%m.%Y, %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  moment_konca = pd.to_datetime(df["end_time"])


In [28]:
df.sort_values(by="start_date", inplace=True)
df.reset_index(drop=True, inplace=True)


In [29]:
df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])

df2025 = df[df["start_date"].dt.year == 2025]



In [30]:
print(len(df[df["start_date"].dt.year == 2026]))
#df[df["start_date"].dt.year == 2026]

20


Ogarnijmy dystans

In [41]:
from distance import dystans 


In [32]:
dystans(df2025["start_place"][33],df2025["end_place"][33]) # nie ma sensu trzeba znalezc baze danych stacji veturilo i porownac z tymi danymi, bo google maps nie rozpoznaje tych nazw stacji

'Brak'

In [33]:
from kordy import kordy
coords = kordy(r"C:\Users\merfc\Downloads\kordy_vet.txt")

In [35]:
df2025["start_coords"] = df2025["start_place"].map(coords.set_index("Station_Name")["Coordinates"])
df2025["end_coords"] = df2025["end_place"].map(coords.set_index("Station_Name")["Coordinates"])


In [ ]:
dystans(df2025["start_coords"][6],df2025["end_coords"][6])

range(0, 49)

for i in range(len(df2025)):
    df2025["dystans"][i] = dystans(df2025["start_coords"][i],df2025["end_coords"][i])
df2025
